# Export

The exporters write slices, obstructions and Smoke3D data to raw binary files with a YAML
file per object describing them, for tools outside Python. They need the export extra:
`pip install "fdsreader[export]"`.

> **The exported values are 8 bit.** Each value is scaled to 0–255 between `DataValMin` and
> `DataValMax` of its object, so the files are a compact copy for visualisation, not a
> lossless one.

Run FDS on `fds_data/input_steckler.fds` first: the Steckler room with a slice, boundary
data and Smoke3D output.

In [1]:
from pathlib import Path

import numpy as np
import yaml

import fdsreader as fds
import fdsreader.export

sim = fds.Simulation("./fds_data")
sim

Simulation(chid=Steckler,
           meshes=1,
           obstructions=4,
           slices=1,
           data_3d=5,
           smoke_3d=3,
           devices=6)

## Export everything

`export_sim` writes all slices, obstructions and Smoke3D data, plus an index file
`CHID-smv.yaml`. `ordering` sets whether the arrays are written in C or Fortran order.

In [2]:
out = Path("fds_post")
index_file = fds.export.export_sim(sim, str(out), ordering="C")

index = yaml.safe_load(open(index_file))
print(yaml.safe_dump({key: index[key] for key in ("Slices", "Obstructions", "Volumes")}))

Obstructions:
- obst/obst-OBST-1_from-hole-1.yaml
- obst/obst-OBST-1_from-hole-2.yaml
- obst/obst-OBST-1_from-hole-3.yaml
- obst/obst-OBST-1_from-hole-4.yaml
Slices:
- slices/temperature/slice2D-.yaml
Volumes:
- smoke/hrrpuv/smoke-hrrpuv.yaml
- smoke/effective_flame_temperature/smoke-effective_flame_temperature.yaml



Single objects can be exported with `export_slcf_raw`, `export_obst_raw` and
`export_smoke_raw` as well.

## Read an exported slice back

The YAML file has the value range, the scale factor and, per mesh, the data file, its size
`(time, x, y, z)` and the grid spacing (the first entry is the time step).

In [3]:
meta = yaml.safe_load(open(out / index["Slices"][0]))
mesh = meta["Meshes"][0]
print({key: meta[key] for key in ("Quantity", "DataValMin", "DataValMax", "ScaleFactor")})
print(mesh)

{'Quantity': 'TEMPERATURE', 'DataValMin': 21.92930030822754, 'DataValMax': 661.7744750976562, 'ScaleFactor': 0.39853391108860015}
{'DataFile': 'slice2D--data/slice2D-_mesh-MESH_0000001.dat', 'DimSize': '101 37 1 23', 'Mesh': 'MESH_0000001', 'MeshPos': '0.0 0.0 0.0', 'Spacing': '1.08228 0.1 0.1 0.09682'}


In [4]:
shape = [int(n) for n in mesh["DimSize"].split()]
raw = np.fromfile(out / Path(index["Slices"][0]).parent / mesh["DataFile"], dtype=np.uint8).reshape(shape)
values = raw / meta["ScaleFactor"] + meta["DataValMin"]

original = sim.slices[0][sim.meshes[0]].data
step = (meta["DataValMax"] - meta["DataValMin"]) / 255
print("exported:", values.shape, " original:", original.shape)
print(f"largest difference: {np.abs(values.squeeze() - original).max():.2f} C, one 8-bit step: {step:.2f} C")

exported: (101, 37, 1, 23)  original: (101, 37, 23)
largest difference: 2.51 C, one 8-bit step: 2.51 C


The values are truncated to the 8-bit scale, so the difference is up to one step.

More in the [API reference](https://fdsreader.readthedocs.io/en/latest/api/export.html).